# 步距Q1&圖.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import pandas as pd
from google.colab import files
import io
import matplotlib.pyplot as plt


In [ ]:
uploaded = files.upload()

all_data = pd.DataFrame()


In [ ]:
file_avg_step_lengths = {}
file_data_counts = {}


In [ ]:
file_q1_step_lengths = {}
file_step_extremes = {}
file_step_ranges = {}
total_file_count = 0


In [ ]:
file_q1_step_lengths = {}
file_step_extremes = {}
file_step_ranges = {}
total_file_count = 0

for file_name in uploaded.keys():

    print(f"Reading file: {file_name}")


    df = pd.read_csv(io.BytesIO(uploaded[file_name]))


    print(f"檢查文件內容: {df.head()}")


    if 'kpts' not in df.columns:
        print(f"Skipping file {file_name}, 'kpts' column not found.")
        continue


    kpts_columns = df['kpts'].str.split(',', expand=True)


    for i in range(kpts_columns.shape[1]):
        kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


    df = df.join(kpts_columns, rsuffix='_kpts')


    print(f"檢查分割後的數據: {df.head()}")


    def step_x(x1, x2):
        return abs(x2 - x1)


    try:
        x16 = pd.to_numeric(df['kpt_46'], errors='coerce')
        x17 = pd.to_numeric(df['kpt_49'], errors='coerce')
    except KeyError:
        print(f"Skipping file {file_name}, 'kpt_46' or 'kpt_49' column not found.")
        continue


    valid_indices = x16.notnull() & x17.notnull()
    x16 = x16[valid_indices]
    x17 = x17[valid_indices]


    step_lengths = [step_x(x16.iloc[i], x17.iloc[i]) for i in range(len(x16))]


    q1_step_length = pd.Series(step_lengths).quantile(0.25) if step_lengths else 0


    min_step_length = min(step_lengths) if step_lengths else 0
    max_step_length = max(step_lengths) if step_lengths else 0
    step_range = max_step_length - min_step_length


    file_q1_step_lengths[file_name] = q1_step_length
    file_step_extremes[file_name] = {'min': min_step_length, 'max': max_step_length}
    file_step_ranges[file_name] = step_range


    total_file_count += 1


    all_data = pd.concat([all_data, df], ignore_index=True)


In [ ]:
total_q1_step_length = sum(file_q1_step_lengths.values()) / total_file_count if total_file_count > 0 else 0
min_step = min(file_step_extremes[file]['min'] for file in file_step_extremes)
max_step = max(file_step_extremes[file]['max'] for file in file_step_extremes)
step_range_total = max_step - min_step


In [ ]:
print(f"Q1總步距變化: {total_q1_step_length}")
print(f"最小步距變化: {min_step}")
print(f"最大步距變化: {max_step}")
print(f"步距變化的極距: {step_range_total}")


In [ ]:
output_csv_filename = "步距Q1數據.csv"
all_data.to_csv(output_csv_filename, index=False)
files.download(output_csv_filename)


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(file_q1_step_lengths.keys(), file_q1_step_lengths.values(), color='skyblue')
plt.title('每個文件的步距 Q1')
plt.xlabel('文件')
plt.ylabel('Q1 步距')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot(file_q1_step_lengths.values(), vert=True, patch_artist=True, labels=['Q1 步距'])
plt.title('步距 Q1 的盒鬚圖')
plt.ylabel('Q1 步距')
plt.show()
